## accounts_user 유저 테이블 전처리

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

display(df.head())

/Users/youju/anaconda3/envs/pythonProject/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,831956,1,1,NaN,600,"[1292473, 913158, 1488461, 1064695, 1043565, 1...",0,2023-03-29 03:44:14.047130+00:00,[],[],N,0,0,0,0,<NA>
1,831962,0,0,F,2248,"[833025, 832642, 982531, 879496, 838541, 83752...",1,2023-03-29 05:18:56.162368+00:00,[],[],N,253,40878,5499,110,12
2,832151,0,0,M,1519,"[838785, 982531, 882567, 879496, 838541, 83649...",0,2023-03-29 12:56:34.989468+00:00,[],[],N,0,37,0,47,1
3,832340,0,0,F,57,"[841345, 982531, 838785, 963714, 882567, 83252...",1,2023-03-29 12:56:35.020790+00:00,[],[],N,0,19,0,21,1
4,832520,0,0,M,1039,"[874050, 849763, 874212, 844297, 838541, 84004...",0,2023-03-29 12:56:35.049311+00:00,[],[],N,0,29,0,15,12


## 결측치 및 데이터 정보 확인

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677085 entries, 0 to 677084
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   id                  677085 non-null  Int64              
 1   is_superuser        677085 non-null  Int64              
 2   is_staff            677085 non-null  Int64              
 3   gender              677083 non-null  str                
 4   point               677085 non-null  Int64              
 5   friend_id_list      677085 non-null  str                
 6   is_push_on          677085 non-null  Int64              
 7   created_at          677085 non-null  datetime64[us, UTC]
 8   block_user_id_list  677085 non-null  str                
 9   hide_user_id_list   677085 non-null  str                
 10  ban_status          677085 non-null  str                
 11  report_count        677085 non-null  Int64              
 12  alarm_count         677085 

In [7]:
df.isna().sum()

id                    0
is_superuser          0
is_staff              0
gender                2
point                 0
friend_id_list        0
is_push_on            0
created_at            0
block_user_id_list    0
hide_user_id_list     0
ban_status            0
report_count          0
alarm_count           0
pending_chat          0
pending_votes         0
group_id              3
dtype: int64

- gender, group_id 결측 존재

In [9]:
missing_rows = df[df.isnull().any(axis=1)]
display(missing_rows)

,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,831956,1,1,NaN,600,"[1292473, 913158, 1488461, 1064695, 1043565, 1...",0,2023-03-29 03:44:14.047130+00:00,[],[],N,0,0,0,0,<NA>
138959,995177,0,0,F,600,[],1,2023-05-08 07:25:51.737820+00:00,[],[],N,0,0,0,0,<NA>
674052,1580689,0,1,NaN,0,[],0,2023-09-24 17:39:12.897884+00:00,[],[],N,0,0,0,0,<NA>


- 677,085행 중 단 3행만 결측치라면 데이터 손실 걱정 없이 그냥 삭제하는 것이 훨씬 효율적이고 안전할 것이라고 판단
- 학급 id부분이 결측이라 그냥 삭제하는 게 나을 것 같음

In [21]:
# 전처리 수행 대상 테이블 호출 (결측치인 행을 SQL에서 직접 제외)
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
    WHERE gender IS NOT NULL 
      AND group_id IS NOT NULL
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

display(df.head())

/Users/youju/anaconda3/envs/pythonProject/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,831962,0,0,F,2248,"[833025, 832642, 982531, 879496, 838541, 83752...",1,2023-03-29 05:18:56.162368+00:00,[],[],N,253,40878,5499,110,12
1,832151,0,0,M,1519,"[838785, 982531, 882567, 879496, 838541, 83649...",0,2023-03-29 12:56:34.989468+00:00,[],[],N,0,37,0,47,1
2,832340,0,0,F,57,"[841345, 982531, 838785, 963714, 882567, 83252...",1,2023-03-29 12:56:35.020790+00:00,[],[],N,0,19,0,21,1
3,832520,0,0,M,1039,"[874050, 849763, 874212, 844297, 838541, 84004...",0,2023-03-29 12:56:35.049311+00:00,[],[],N,0,29,0,15,12
4,832614,0,0,M,1048,"[838541, 833041, 832151, 837806, 1437874, 1142...",1,2023-03-29 12:56:35.064406+00:00,[],[],N,0,28,0,14,12


## 중복값

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df['id'].duplicated().sum()

np.int64(0)

## 이상치

- 전체적으로 이상치 제거하지 않고 그대로 진행. 각자 진행하시는 분석에 따라서 아래 마크다운 참고해주세요

In [13]:
df.describe()

,id,is_superuser,is_staff,point,is_push_on,report_count,alarm_count,pending_chat,pending_votes,group_id
count,677082.0,677082.0,677082.0,677082.0,677082.0,677082.0,677082.0,677082.0,677082.0,677082.0
mean,1212969.466533,0.000001,0.000001,3039.156531,0.8431,0.037291,0.94665,0.09847,84.630893,37022.980168
std,213896.3941,0.001215,0.001215,1076022.326839,0.363706,0.588106,56.114983,11.087794,123.262832,21997.765398
min,831962.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0
25%,1028076.25,0.0,0.0,400.0,1.0,0.0,0.0,0.0,2.0,18488.0
50%,1211729.5,0.0,0.0,965.0,1.0,0.0,1.0,0.0,29.0,35614.0
75%,1397905.75,0.0,0.0,2183.0,1.0,0.0,1.0,0.0,122.0,54534.0
max,1583733.0,1.0,1.0,885000006.0,1.0,253.0,40878.0,5712.0,3352.0,84546.0


- point : 75% 수준이 2,183인 것에 비해 최대값이 너무 큼(885000006.0)
- - 일단 지우지 않고 진행
- alarm_count도 비슷
- - 2~3행정도 이상치라고 판단되는데 이것도 alarm_count 컬럼 사용하시는 분들 참고하시면 좋을 것 같습니다!
- pending_chat, pending_votes 
- - 동일

- is_superuser이거나 is_Staff일 가능성이 있다고 생각했는데 각각 1명씩만 잇고 이상치 컬럼에 해당되지 않는 걸 보면 단순 이상치가 맞는 것 같습니다

## 파생변수

- block_user_id_list, hide_user_id_list, friend_id_list 컬럼
- - str형 리스트값으로 저장되어 있음

- - 리스트 원소 수를 센 값을 파생변수로써 지정
- - friend_count, block_count, hide_count

In [31]:
# 전처리 수행 대상 테이블 호출 및 SQL 내에서 리스트 개수 컬럼 생성
sql = f"""
    SELECT 
        *,
        -- 빈 리스트 '[]'인 경우 0, 아니면 콤마 개수 + 1로 친구 수 계산
        CASE 
            WHEN friend_id_list = '[]' THEN 0
            ELSE (LENGTH(friend_id_list) - LENGTH(REPLACE(friend_id_list, ',', ''))) + 1
        END AS friend_count,
        
        CASE 
            WHEN block_user_id_list = '[]' THEN 0
            ELSE (LENGTH(block_user_id_list) - LENGTH(REPLACE(block_user_id_list, ',', ''))) + 1
        END AS block_count,

        CASE 
            WHEN hide_user_id_list = '[]' THEN 0
            ELSE (LENGTH(hide_user_id_list) - LENGTH(REPLACE(hide_user_id_list, ',', ''))) + 1
        END AS hide_count

    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
    WHERE gender IS NOT NULL 
      AND group_id IS NOT NULL
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

display(df.head())


/Users/youju/anaconda3/envs/pythonProject/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id,friend_count,block_count,hide_count
0,831962,0,0,F,2248,"[833025, 832642, 982531, 879496, 838541, 83752...",1,2023-03-29 05:18:56.162368+00:00,[],[],N,253,40878,5499,110,12,43,0,0
1,832151,0,0,M,1519,"[838785, 982531, 882567, 879496, 838541, 83649...",0,2023-03-29 12:56:34.989468+00:00,[],[],N,0,37,0,47,1,51,0,0
2,832340,0,0,F,57,"[841345, 982531, 838785, 963714, 882567, 83252...",1,2023-03-29 12:56:35.020790+00:00,[],[],N,0,19,0,21,1,57,0,0
3,832520,0,0,M,1039,"[874050, 849763, 874212, 844297, 838541, 84004...",0,2023-03-29 12:56:35.049311+00:00,[],[],N,0,29,0,15,12,18,0,0
4,832614,0,0,M,1048,"[838541, 833041, 832151, 837806, 1437874, 1142...",1,2023-03-29 12:56:35.064406+00:00,[],[],N,0,28,0,14,12,21,0,0
